# {{来源}}数据清洗 — 逐类确认

共享来源规则与确认记录放在 `notebooks/数据清洗/`，供场景引用。
这份 notebook 按问题分别说明影响、完整明细、建议和业务结论。

- 复制时替换字段、窗口、数值容差和来源读取函数，补齐计算占位；检查读取接口已经实现。
- 示例读取函数遵循目标约定，不用旧的自动换层入口代替。
- 原始 dataframe 保留不动，处置在副本预演；未确认规则不执行。
- 完整明细保留为命名 dataframe，输出格只放全量汇总或字段说明，不截断、不导出明细。

In [ ]:
import pandas as pd
from read_data import read_raw_procurement_inbound

# 示例来源，复制时换成实际已实现的 read_raw_<来源>
df = read_raw_procurement_inbound("{{起始周期}}", "{{结束周期}}")
RAW_DATE_COLUMN = "{{原始日期列}}"
RAW_AMOUNT_COLUMN = "{{原始金额列}}"
STANDARD_AMOUNT_COLUMN = "{{标准金额列}}"
RAW_TO_STANDARD_AMOUNT = float("{{原始金额转标准单位系数，同单位填1}}")
AMOUNT_ATOL = float("{{业务确认的标准金额绝对容差}}")
# 解析用于诊断的日期，不覆盖 raw；失败记录单独确认，不能读取时静默剔除。
parsed_dates = pd.to_datetime(df[RAW_DATE_COLUMN], errors="coerce")
invalid_date_rows = df.loc[parsed_dates.isna()].copy()

pd.DataFrame([{
    "行数": len(df), "列数": df.shape[1],
    "原始金额合计": pd.to_numeric(df[RAW_AMOUNT_COLUMN], errors="raise").sum(min_count=1),
    "日期解析失败行数": len(invalid_date_rows),
    "日期范围": f"{parsed_dates.min()} ~ {parsed_dates.max()}",
}])

## 一、先给每列做个体检

看看每一列缺了多少、有多少种取值。排在前面的列问题最大，也是后面逐节重点查的对象。

In [ ]:
# 全列画像，不裁掉缺失严重或暂未用于本场景的字段。
profile = pd.DataFrame({
    "类型": df.dtypes.astype(str),
    "非空率": df.notna().mean(),
    "取值种数": df.nunique(),
})
profile.sort_values(["非空率", "取值种数"])

**体检结论**（跑完上面一格后回填）：{{哪些列有什么问题}}。下面按严重程度逐类来看。

---

## 二、{{异常类}}

**怎么认出来的**：{{用大白话说清什么样的数据算这类问题，点到具体字段}}

In [ ]:
# 示例是重复候选，不代表应删除；按当前问题替换判定表达式。
mask = df.duplicated(keep=False)
issue_rows = df.loc[mask].copy()  # 全量问题明细，供 Data Wrangler 翻查

pd.DataFrame([{
    "问题行数": len(issue_rows),
    "占比": float(mask.mean()) if len(df) else 0.0,
    "涉及原始金额": pd.to_numeric(issue_rows[RAW_AMOUNT_COLUMN], errors="raise").sum(min_count=1),
}])

In [ ]:
# 保留日期解析失败分组，不能因 groupby 默认丢空值而隐藏异常。
issue_months = parsed_dates.loc[mask].dt.strftime("%Y-%m")
issue_by_month = issue_rows.groupby(issue_months, dropna=False).size().to_frame("问题行数")
issue_by_month

In [ ]:
# 完整问题明细保留在 issue_rows；输出只显示全量分组汇总。
issue_summary = (
    issue_rows.groupby(["{{单号列}}", "{{物料列}}"], dropna=False)
    .agg(行数=(RAW_AMOUNT_COLUMN, "size"), 原始金额=(RAW_AMOUNT_COLUMN, lambda s: s.sum(min_count=1)))
    .sort_values("原始金额", ascending=False)
)
print(f"问题明细变量 issue_rows，共 {len(issue_rows):,} 行；定位条件：{{本节判定条件}}")
issue_summary

**建议怎么处理**：{{修格式 / 这部分先不参与统计 / 保留但打上标记}} —— {{一句话理由}}

**你的结论**：{{同意 / 不同意，改成 ___}} —— 你定了我们再看下一类

---

{{「一节 + 四格」照这个样子按问题类型复制，编号往后排}}

## 最后、预演已确认的处理并对总账

每个处置必须已有业务结论；阻塞金额、标准字段或分析结论的待定项先解决。
无影响的待定项保留原值并标注，不提供未确认的删行开关。

下例假设金额仅做标准单位转换，不修正业务金额。若业务确认需改金额，先补净调整台账再扩展勾稽。
剔除明细必须是实际剔除的原始行，各项不重复计数。

In [ ]:
def apply_confirmed_rules(raw):
    # 按本 notebook 逐项确认的规则填写；返回标准明细和实际剔除的原始明细。
    raise NotImplementedError("填写已确认清洗链，返回 (clean, excluded_rows)")

clean, excluded_rows = apply_confirmed_rules(df.copy())

def amount_total(frame, column):
    return pd.to_numeric(frame[column], errors="raise").sum(min_count=1) if len(frame) else 0.0

before_amount = amount_total(df, RAW_AMOUNT_COLUMN) * RAW_TO_STANDARD_AMOUNT
excluded_amount = amount_total(excluded_rows, RAW_AMOUNT_COLUMN) * RAW_TO_STANDARD_AMOUNT
after_amount = amount_total(clean, STANDARD_AMOUNT_COLUMN)
row_difference = len(df) - len(clean) - len(excluded_rows)
amount_difference = before_amount - after_amount - excluded_amount

audit = pd.DataFrame([
    {"项": "清洗前", "行数": len(df), "标准单位金额": before_amount},
    {"项": "实际剔除", "行数": len(excluded_rows), "标准单位金额": excluded_amount},
    {"项": "清洗后", "行数": len(clean), "标准单位金额": after_amount},
    {"项": "勾稽差异", "行数": row_difference, "标准单位金额": amount_difference},
])
assert row_difference == 0, "行数对不上，先查实际剔除及重复计数"
assert pd.notna(amount_difference) and abs(amount_difference) <= AMOUNT_ATOL, "金额对不上或缺失，先查根因"
audit

In [ ]:
# 完整标准字段、类型与质量画像，按已确认 schema 核对，不截断。
processed_profile = pd.DataFrame({
    "类型": clean.dtypes.astype(str),
    "非空率": clean.notna().mean(),
    "取值种数": clean.nunique(),
})
processed_profile.sort_values("非空率")

## 收尾：完整标准明细确认后，在第三步沉淀并归档

完整明细保留为 `processed_rows`，请用 Data Wrangler / 变量查看器逐列翻查。
输出格只显示行数与金额摘要，不渲染全表、不导 Excel。
确认后在本步沉淀稳定清洗函数与入口，读回 processed 与预演逐项对拍，不等到第四步才固化。

In [ ]:
processed_rows = clean
print(f"完整标准明细变量 processed_rows，共 {len(processed_rows):,} 行 × {processed_rows.shape[1]} 列")
pd.DataFrame([{
    "行数": len(processed_rows),
    "标准单位金额": amount_total(processed_rows, STANDARD_AMOUNT_COLUMN),
}])

**确认与落地记录**：

- [ ] 完整标准明细字段、粒度、类型、单位、行数与金额已确认
- [ ] 待定项已列明，不执行未确认处置；阻塞项未解决则不宣布底表可用
- [ ] 清洗函数接收 dataframe，读写由按来源/数据集命名的入口串联
- [ ] 本步完成 processed 归档，记录输入文件版本/指纹、规则版本与覆盖
- [ ] 用显式 `read_processed_<数据集>()` 读回，与预演核对字段、类型、关键明细和分组账
- [ ] 共享来源文档记录规则与确认，场景 3.4 引用命令、版本、归档和对拍结论

第四步只读上述已归档 processed，不重放清洗链。

**待定项**：{{问题 / 当前保留或标注方式 / 是否阻塞底表可用 / 后续确认依据}}。

已有合格 processed 可直接被其他场景复用；新增场景不能据自己的选列清单裁掉共享字段。